# PPO Mine Scheduling — Walkthrough

1. Generate MineLib instance
2. Explore the environment
3. Inspect the Actor-Critic network
4. Train PPO
5. Compare vs baselines


In [ ]:
import sys
sys.path.insert(0, '../src')
import numpy as np
import torch
import matplotlib.pyplot as plt
from instance_loader import generate_instance, save_instance, greedy_npv_sequence
from environment import MineSchedulingEnv
from network import ActorCriticNetwork, count_parameters
from ppo_agent import PPOAgent, Trajectory
from train import train, CONFIG
import copy
print('Ready')

## Step 1: Generate a MineLib-format Instance

In [ ]:
inst = generate_instance(name='demo', n_benches=3, n_rows=4, n_cols=4, n_periods=3, seed=42)
print(f'Instance: {inst.name}')
print(f'Blocks: {inst.n_blocks}  |  Periods: {inst.n_periods}')
print(f'Capacity: {inst.capacity_tonnes:.0f} t/period')
print(f'Ore blocks: {sum(1 for b in inst.blocks if b["is_ore"])}')
print(f'Waste blocks: {sum(1 for b in inst.blocks if not b["is_ore"])}')
print(f'Total NPV potential: {sum(b["npv"] for b in inst.blocks if b["is_ore"]):.2f}')

## Step 2: Explore the Environment

In [ ]:
env = MineSchedulingEnv(inst)
state, avail = env.reset()
print(f'State dimension: {len(state)}')
print(f'Available blocks at start: {np.where(avail)[0].tolist()}')
print(f'Greedy NPV: {env.greedy_rollout():.2f}')

## Step 3: Actor-Critic Network

In [ ]:
net = ActorCriticNetwork(state_dim=env.state_dim, n_blocks=env.n_blocks, hidden_dim=128)
print(f'Parameters: {count_parameters(net):,}')
s = torch.tensor(state).unsqueeze(0)
m = torch.tensor(avail, dtype=torch.bool).unsqueeze(0)
dist, val = net(s, m)
print(f'Value estimate: {val.item():.4f}')
print(f'Top-3 action probs: {dist.probs[0].topk(3).values.tolist()}')

## Step 4: Train PPO

In [ ]:
nb_config = copy.deepcopy(CONFIG)
nb_config['instance_path'] = '../data/small_instance.json'
nb_config['output_dir']   = '../outputs'
nb_config['model_path']   = '../outputs/ppo_mine.pth'
nb_config['plot_path']    = '../outputs/training_curve.png'
nb_config['n_episodes']   = 500
nb_config['print_every']  = 100
agent = train(nb_config)

In [ ]:
from IPython.display import Image
Image('../outputs/training_curve.png')